# run_all_0416b_nb12 — adaptive seller × adaptive buyer (both adaptive)

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **6 adaptive seller schedules (mix of 2- and 3-stage)**
* Buyer strategies covered  : **6 adaptive buyer profiles**
* Mode                      : **both adaptive**
* Target per-notebook runtime : **~16 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 12000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb12"


## EXPERIMENTS list — notebook 12

In [2]:
# Both adaptive: 6 adaptive buyers × 6 adaptive seller schedules × 6 runs
# = 216 runs (~16.2 h).

def _ab(name, subs, sched):
    return AdaptiveBuyer(substrats=subs, schedule=sched, label=name)

ADAPTIVE_BUYERS = [
    _ab("ab_anchor_to_concession",
        [AnchorDragBuyer(), ConcessionBuyer()],
        [8]),
    _ab("ab_wild_to_boulware",
        [WildBuyer(), BoulwareBuyer()],
        [8]),
    _ab("ab_budget_to_tft",
        [BudgetBuyer(), TitForTatBuyer()],
        [8]),
    _ab("ab_persistent_tft_concession",
        [PersistentBuyer(0.30), TitForTatBuyer(), ConcessionBuyer()],
        [6, 11]),
    _ab("ab_anchor_conceder_concession",
        [AnchorDragBuyer(), ConcederBuyer(), ConcessionBuyer()],
        [6, 11]),
    _ab("ab_random_concession_tft",
        [RandomBuyer(), ConcessionBuyer(), TitForTatBuyer()],
        [6, 11]),
]

ADAPTIVE_SELLER_SCHEDULES = [
    twostage_seller_schedule("anchor_high",       "cooperative",         8, N_ROUNDS),
    twostage_seller_schedule("boulware_seller",   "conceder_seller",     8, N_ROUNDS),
    twostage_seller_schedule("cooperative",       "anchor_high",         8, N_ROUNDS),
    threestage_seller_schedule("anchor_high",     "cooperative",
                               "conceder_seller",  6, 11, N_ROUNDS),
    threestage_seller_schedule("boulware_seller", "tit_for_tat_matcher",
                               "conceder_seller",  6, 11, N_ROUNDS),
    threestage_seller_schedule("relational",      "cooperative",
                               "boulware_seller",  6, 11, N_ROUNDS),
]

N_RUNS_PER_PAIR = 6

EXPERIMENTS = []
for sched in ADAPTIVE_SELLER_SCHEDULES:
    for ab in ADAPTIVE_BUYERS:
        tag = f"sel_{sched.label}__buy_{ab.name}"
        EXPERIMENTS.append({
            "tag": tag, "buyer": ab, "seller_schedule": sched,
            "n_runs": N_RUNS_PER_PAIR,
        })

print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 36
Total runs      : 216


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/36]  sel_anchor_high__then__cooperative@r8__buy_ab_anchor_to_concession
  buyer   = ab_anchor_to_concession
  seller  = anchor_high__then__cooperative@r8  (adaptive=True)
  n_runs  = 6   n_rounds = 15

  --- Run 1/6  seed=17000  prod=Power bank (5000mAh) cost=$11.58 mkt=$40.00 int=$23.20 ---
  R 1 BUY $12.89 T0 [buy=anchor_drag]  SEL $40.00 [sel=anchor_high]  "Thank you for your interest. However, the market value for a..."
  R 2 BUY $13.82 T3 [buy=anchor_drag]  SEL $38.50 [sel=anchor_high]  "I understand you're looking for a competitive rate, but $13...."
  R 3 BUY $14.89 T5 [buy=anchor_drag]  SEL $37.50 [sel=anchor_high]  "I appreciate your continued interest, but I must emphasize t..."
  R 4 BUY $15.45 T3 [buy=anchor_drag]  SEL $36.50 [sel=anchor_high]  "I appreciate your persistence, but we are still very far apa..."
  R 5 BUY $16.29 T2 [buy=anchor_drag]  SEL $36.00 [sel=anchor_high]  "I appreciate your updated offer, but $16.29 per unit still f..."
  R 6 BUY $17.55 T3 [buy=anc

## Inspection

In [4]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 5087: illegal multibyte sequence